## Set up

In [1]:
%pip install neo4j
%pip install matplotlib


[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Imports

In [2]:
from neo4j import GraphDatabase

#### Load classes

In [ ]:
%run ../commons/models.py

### Establish connection and create driver

In [3]:
uri = "bolt://0.0.0.0:7687"
username = "neo4j"
password = "111122223333"
driver = GraphDatabase.driver(uri, auth=(username, password))

#### Get the image data

In [5]:
# query = """
#     MATCH (n)
#     WHERE NOT "Pixel" IN labels(n) AND NOT "Line" IN labels(n)
#     RETURN n
# """
query = """
    MATCH (target {image_id: '7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47'})-[*0..5]-(connectedNode)
    WHERE NOT "Line" IN labels(connectedNode)
    RETURN connectedNode
"""



with driver.session() as session:
    result = session.execute_read(execute_query, query)
    for record in result:
        print(record)

#### Clean up

In [6]:
def execute_query(tx, query):
    result = tx.run(query)
    return [record for record in result]

In [11]:
clean_up_query = """
    MATCH (pixel:Pixel)
    DETACH DELETE pixel
"""

with driver.session() as session:
    result = session.execute_write(execute_query, clean_up_query)
    for record in result:
        print(record)

Failed to write data to connection IPv4Address(('0.0.0.0', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687)))
Transaction failed and will be retried in 0.933526830057869s (Failed to write data to connection IPv4Address(('0.0.0.0', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))))


#### Playground

In [ ]:
clean_up_query = """
    MATCH path=(cp: CriticalPoint {reason: "First point"})--()
    RETURN path
"""

with driver.session() as session:
    result = session.execute_read(execute_query, clean_up_query)
    for record in result:
        print(record)

In [43]:
# %load ../commons/models.py
from dataclasses import dataclass
from enum import Enum

class Reason(Enum):
    FIRST_POINT = "First point"
    FIRST_LINE = "First Line"

@dataclass(frozen=True)
class CriticalPoint:
    uuid: str
    image_id: str
    reason: Reason
    connectedVerts = {}

In [46]:
def get_starting_point(results):
    critical_points = set()
    first_points = set()
    first_lines = set()
    
    for record in results:
        record = record['connectedNode']
        label = [x for x in record.labels][0]
        if label == "CriticalPoint":
            critical_point = CriticalPoint(uuid=record['uuid'], reason=record['reason'], image_id=record['image_id'])
            critical_points.add(critical_point)
            
    for value in critical_points:
        if value.reason == Reason.FIRST_POINT.value:
            first_points.add(value)
        if value.reason == Reason.FIRST_LINE.value:
            first_lines.add(value)
            
    return first_points, first_lines

In [ ]:
def comapare_nodes(node1, node2):
    if isinstance(node1, CriticalPoint):
        return compare_critical_points(node1, node2)
    else:
        return "Unsupported node type"
    
    
def compare_critical_points(cp1: CriticalPoint, cp2: CriticalPoint):
    return 1 if (cp1.reason == cp2.reason) else 0

In [49]:
first_points, first_lines = get_starting_point(result)
first_critical_points = {critical_point.image_id: critical_point for critical_point in first_points}
first_critical_lines = {critical_point.image_id: critical_point for critical_point in first_lines}
print(f"First points: {first_critical_points}")
print(f"First lines: {first_critical_lines}")

with driver.session() as session:
    go_over_contour_query = """
        MATCH (cp1:CriticalPoint {uuid: })
    """
    session.execute_read(execute_query, )

First points: {'7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47': CriticalPoint(uuid='8db2ae58-2358-43ed-a872-51e0b97750c6', image_id='7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47', reason='First point')}
First lines: {'7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47': CriticalPoint(uuid='92a27982-70b4-4aca-ba58-f07546a97cc1', image_id='7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47', reason='First Line')}


#### Close the driver

In [ ]:
driver.close()